In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import accuracy_score, confusion_matrix, precision_score, recall_score, f1_score, roc_curve, auc, precision_recall_curve
import matplotlib.pyplot as plt
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import SimpleRNN, Dense

# Load the dataset
def load_data():
    # Replace with the appropriate path to your dataset files
    train_data = pd.read_csv("KDDTrain+.txt", header=None)
    test_data = pd.read_csv("KDDTest+.txt", header=None)
    return train_data, test_data

# Preprocess the dataset
def preprocess_data(data):
    # Encode categorical columns using LabelEncoder
    label_encoder = LabelEncoder()
    for col in data.select_dtypes(include=['object']).columns:
        data[col] = label_encoder.fit_transform(data[col])
    
    # Separate features (X) and labels (y)
    X = data.iloc[:, :-1].values  # Features (all columns except last)
    y = data.iloc[:, -1].values   # Labels (last column)
    
    # Standardize features using StandardScaler
    scaler = StandardScaler()
    X = scaler.fit_transform(X)
    
    return X, y

# Split the dataset into training, validation, and test sets
def split_data(X_train_full, y_train_full):
    X_train, X_val, y_train, y_val = train_test_split(X_train_full, y_train_full, test_size=0.2, random_state=42)
    return X_train, X_val, y_train, y_val

# Build the RNN model
def build_rnn(input_shape):
    model = Sequential()
    model.add(SimpleRNN(64, input_shape=input_shape))
    model.add(Dense(32, activation='relu'))
    model.add(Dense(1, activation='sigmoid'))  # Binary classification
    
    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
    return model

# Plot confusion matrix
def plot_confusion_matrix(cm):
    plt.figure(figsize=(6, 6))
    plt.imshow(cm, interpolation='nearest', cmap=plt.cm.Blues)
    plt.title("Confusion Matrix")
    plt.colorbar()
    tick_marks = np.arange(2)
    plt.xticks(tick_marks, ["Normal", "Attack"], rotation=45)
    plt.yticks(tick_marks, ["Normal", "Attack"])
    
    thresh = cm.max() / 2.0
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            plt.text(j, i, format(cm[i, j], 'd'), horizontalalignment="center",
                     color="white" if cm[i, j] > thresh else "black")
    
    plt.ylabel('True label')
    plt.xlabel('Predicted label')
    plt.tight_layout()
    plt.show()

# Plot ROC curve
def plot_roc_curve(y_true, y_pred):
    fpr, tpr, _ = roc_curve(y_true, y_pred)
    roc_auc = auc(fpr, tpr)
    
    plt.figure(figsize=(8, 6))
    plt.plot(fpr, tpr, color='darkorange', lw=2,
             label=f"ROC curve (area = {roc_auc:.2f})")
    plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.title('Receiver Operating Characteristic (ROC)')
    plt.legend(loc="lower right")
    plt.show()

# Plot Precision-Recall curve
def plot_precision_recall_curve(y_true, y_pred):
    precision, recall, _ = precision_recall_curve(y_true, y_pred)
    
    plt.figure(figsize=(8, 6))
    plt.plot(recall, precision)
    plt.xlabel('Recall')
    plt.ylabel('Precision')
    plt.title('Precision-Recall Curve')
    plt.show()

# Main function to execute the pipeline
if __name__ == "__main__":
    
    # Load and preprocess data
    train_data_full, test_data_full = load_data()
    
    X_train_full, y_train_full = preprocess_data(train_data_full)
    X_test_full, y_test_full = preprocess_data(test_data_full)
    
    X_train, X_val, y_train, y_val = split_data(X_train_full, y_train_full)

    # Build and train the RNN model
    input_shape = (X_train.shape[1],)  # Number of features as input shape
    rnn_model = build_rnn(input_shape)
    
    rnn_model.fit(X_train,
                  y_train,
                  epochs=10,
                  batch_size=32,
                  validation_data=(X_val ,y_val))
    
     # Evaluate on test set
     y_pred_proba = rnn_model.predict(X_test_FULL )
